In [ ]:
import pandas as pd
import os
import shutil
from tqdm import tqdm

BASE_PATH = "./MIQR-CC-DATASET"
CSV_PATH = os.path.join(BASE_PATH, "metadata.csv")
OUTPUT_UNLABELED_DIR = "./dataset/pretrain_unlabeled"

def extract_unlabeled_data():
    if not os.path.exists(CSV_PATH):
        print(f"Error: CSV not found at {CSV_PATH}. Make sure your folder structure is correct.")
        return

    print("Loading metadata...")
    df = pd.read_csv(CSV_PATH)

    # Filter ONLY for Unlabelled AND Keep frames (to avoid duplicate patient frames)
    df_unlabeled = df[(df['Keep'] == 'Keep') & (df['Label'] == 'Unlabelled')].copy()

    if df_unlabeled.empty:
        print("No unlabeled 'Keep' frames found.")
        return

    os.makedirs(OUTPUT_UNLABELED_DIR, exist_ok=True)

    print(f"Found {len(df_unlabeled)} unlabeled images for the pseudo-labeling phase. Copying...")
    
    copied = 0
    missing = 0
    
    for _, row in tqdm(df_unlabeled.iterrows(), total=len(df_unlabeled), desc="Copying Images"):
        # The processed_image_path looks like "processed/1_image1.png"
        src = os.path.join(BASE_PATH, row['processed_image_path'])
        
        dst_name = f"{row['patient_id']}_{os.path.basename(row['processed_image_path'])}"
        dst = os.path.join(OUTPUT_UNLABELED_DIR, dst_name)
        
        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing += 1

    print(f"\nExtraction complete!")
    print(f"Successfully copied: {copied}")
    if missing > 0:
        print(f"Missing source files: {missing} (Check if your processed folder is fully extracted)")

if __name__ == "__main__":
    extract_unlabeled_data()

Loading metadata...
Found 13648 unlabeled images for the pseudo-labeling phase. Copying...


Copying Images: 100%|██████████| 13648/13648 [00:10<00:00, 1252.24it/s]


Extraction complete!
Successfully copied: 13648
